# Where the risk lives

**Can this register be attributed to an owner at all, and which parts of the estate carry
the backlog?**

This is GAS's *Attribution* page, renamed rather than faked. GAS attributes each finding
to a value-chain domain through configurable rules over subscriptions and asset tags, and
then audits how much of the register those rules actually claim.

**brick has neither.** There are no domain rules, and the ingest query selects no asset
tags — so there is nothing here to compute a coverage-gap against. Rather than relabel
`subscription_name` as a value chain and let the page imply an ownership map that does not
exist, it answers the nearest question that is actually answerable: what raw material is
there, and where is the backlog concentrated?

Adding asset tags to `ingest.py` is the real fix. This page is what the register supports
until then.

## How to read this notebook

Every cell answers one question and shows one thing. Run them in order the first time; after
that any cell can be re-run on its own.

**Set the widgets at the top before you run anything.** `catalog` has no default on purpose.
Set the notebook to **Run accessed commands** (the dropdown beside *Run all*) if you want a
widget change to re-run the cells that depend on it — otherwise you will change the filter and
read a chart drawn under the old one.

Everything here reads one scan, pinned in cell 1. Charts that span scans say so in their title.

In [ ]:
PAGE = {"group_by": ("subscription_name", ["subscription_name", "asset_type", "cloud"])}

import os, sys

_paths = []
try:
    _paths.append(dbutils.widgets.get("module_path"))
except Exception:  # noqa: BLE001 -- the widget does not exist yet on a first run
    pass
_here = os.getcwd()
_paths += [_here, os.path.dirname(_here)]
for _p in _paths:
    if _p and os.path.exists(os.path.join(_p, "panels.py")):
        sys.path.insert(0, _p)
        break
else:
    raise RuntimeError("brick modules are not on sys.path -- see brick/README.md, step 2")

import panels, figures, tiles

panels.declare_widgets(**PAGE)
ctx = panels.context(spark, **{name: str(spec[0]) for name, spec in PAGE.items()})
displayHTML(tiles.scan_zone_from(panels.last_scan(spark, ctx).first()))

## Could this be attributed?

For each dimension that *is* captured: how much of the register carries it, and how many
distinct values it takes. Both matter. A dimension that is 100% populated with three
values attributes nothing; one that is 40% populated is not a map.

In [ ]:
display(panels.attributability(spark, ctx))

## Where the backlog sits

`signal_coverage_pct` is the share of each group the high-risk rule could classify at all.
A group with low signal coverage is not low risk — it is unmeasured, and the coverage and
efficiency figures on `02_program_performance` exclude it from both sides.

In [ ]:
display(panels.coverage_by_group(spark, ctx, ctx.param('group_by')))

Chart ▸ Bar (horizontal) · X=group_value · Y=lifecycles · Group by=risk_label ·
Stacking=100% · Legend=bottom · X title=Group · Y title=Share of lifecycles

The risk mix per group, normalised — GAS's proportional strip. Three **named** categories,
deliberately not severity: a stacked bar is precisely the chart where a reader has to tell
two adjacent fills apart, and the severity ramp is the one palette that fails at it.

In [ ]:
display(panels.risk_mix(spark, ctx, ctx.param('group_by')))

Chart ▸ Pivot Table · Rows=group_value · Columns=severity · Value=open · Order=sev_rank

In [ ]:
display(panels.group_severity(spark, ctx, ctx.param('group_by')))